In [3]:
import time
from concurrent.futures import ThreadPoolExecutor, wait
from collections import deque
from typing import List, Iterator, Deque, Tuple, Optional
import random

In [2]:
def fake_raw_dataset(num_items: int) -> List[int]:
    """模拟原始数据，每个整数代表一条文档。"""
    return list(range(num_items))


def batched(iterable, batch_size) -> Iterator[List[int]]:
    """简单的批处理迭代器。"""
    batch = []
    for x in iterable:
        batch.append(x)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


In [5]:
raw_dataset = fake_raw_dataset(10)
raw_dataset

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [8]:
batched_dataset = batched(raw_dataset, batch_size=2)
batched_dataset


<generator object batched at 0x10851f010>

In [11]:
next(batched_dataset)

[4, 5]

In [12]:
def preprocess_batch(batch: List[int]) -> List[str]:
    """
    预处理（模拟 CPU 文本切分）。
    这里我们 sleep 一下表示耗时，然后把每个数字转成 'doc-<n>-chunk-<k>' 的伪 chunk。
    """
    # 模拟不同耗时：比如文本长度不同
    simulated_cost = 0.05 + random.random() * 0.05
    time.sleep(simulated_cost)
    processed = [f"doc-{x}-chunk-0" for x in batch]  # 简化：每条只切 1 个 chunk
    return processed

preprocess_batch(next(batched_dataset))

['doc-6-chunk-0', 'doc-7-chunk-0']

In [ ]:
def preprocess_batch(batch: List[int]) -> List[str]:
    """
    预处理（模拟 CPU 文本切分）。
    这里我们 sleep 一下表示耗时，然后把每个数字转成 'doc-<n>-chunk-<k>' 的伪 chunk。
    """
    # 模拟不同耗时：比如文本长度不同
    simulated_cost = 0.05 + random.random() * 0.05
    time.sleep(simulated_cost)
    processed = [f"doc-{x}-chunk-0" for x in batch]  # 简化：每条只切 1 个 chunk
    return processed


def embed_and_insert(chunks: List[str]):
    """
    模拟 GPU embedding + 向量库写入。
    用 sleep 模拟 GPU/网络时间。
    """
    simulated_cost = 0.06 + random.random() * 0.04
    time.sleep(simulated_cost)
    # 这里只打印一下
    print(f"[Consumer] embedded & inserted {len(chunks)} chunks (example={chunks[0]})")


def submit_prefetch_task(raw_iter: Iterator[List[int]],
                         raw_batches: Deque[List[int]],
                         executor: ThreadPoolExecutor):
    """
    尝试获取下一个 raw batch 并提交预处理任务。
    若耗尽返回 None。
    """
    try:
        nxt = next(raw_iter)
    except StopIteration:
        return None
    raw_batches.append(nxt)
    start_t = time.time()

    def _task(docs: List[int]) -> Tuple[List[str], float]:
        processed = preprocess_batch(docs)
        return processed, time.time() - start_t

    return executor.submit(_task, nxt)


def prefetch_window_pipeline(total_items=40, batch_size=8, prefetch_depth=2):
    """
    核心：保持一个大小为 prefetch_depth 的窗口：
    - 先填满
    - 逐个取出已完成的 future（按提交顺序），消费
    - 每消费 1 个再补 1 个
    """
    dataset = fake_raw_dataset(total_items)
    raw_iter = batched(dataset, batch_size)

    executor = ThreadPoolExecutor(max_workers=prefetch_depth)
    futures: Deque = deque()
    raw_batches: Deque[List[int]] = deque()
    batch_index = 0

    try:
        # 1. 预热窗口
        for _ in range(prefetch_depth):
            fut = submit_prefetch_task(raw_iter, raw_batches, executor)
            if fut is None:
                break
            futures.append(fut)

        # 2. 主循环：不断取出最早的 future
        while futures:
            fut = futures.popleft()
            raw_batch = raw_batches.popleft()
            processed_chunks, preprocess_time = fut.result()  # 阻塞直到该批预处理完成（但其他批此时已在并行跑）
            print(f"[Producer->Consumer] batch {batch_index}: raw={len(raw_batch)} "
                  f"processed={len(processed_chunks)} preprocess_time={preprocess_time:.3f}s")

            # Consumer 阶段：做 embedding + insert
            embed_and_insert(processed_chunks)

            batch_index += 1

            # 3. 补位
            new_fut = submit_prefetch_task(raw_iter, raw_batches, executor)
            if new_fut is not None:
                futures.append(new_fut)

    finally:
        executor.shutdown(wait=True)


if __name__ == \"__main__\":
    print(\"=== Prefetch window demo (prefetch_depth=2) ===\")
    t0 = time.time()
    prefetch_window_pipeline(prefetch_depth=2)
    print(f\"Total elapsed: {time.time() - t0:.2f}s\\n\")

    print(\"=== 对比：无预取（串行）===\")
    # 串行对比：每批先预处理，再 embed
    dataset = fake_raw_dataset(40)
    t1 = time.time()
    for bidx, raw_batch in enumerate(batched(dataset, 8)):
        start_p = time.time()
        chunks = preprocess_batch(raw_batch)
        preprocess_time = time.time() - start_p
        print(f\"[Serial] batch {bidx}: raw={len(raw_batch)} processed={len(chunks)} preprocess_time={preprocess_time:.3f}s\")
        embed_and_insert(chunks)
    print(f\"Total elapsed (serial): {time.time() - t1:.2f}s\")

In [5]:
def fn(prefetch_depth: Optional[int] = None):
    res = []
    for i in range(prefetch_depth):
        res.append(i)
    
    while res is not None:
        res.pop(0)
        yield res

        res.append(10)

In [6]:
res = fn(3)
res

<generator object fn at 0x108482ea0>

In [7]:
next(res)

[1, 2]

In [8]:
next(res)

[2, 10]

In [4]:
executor = ThreadPoolExecutor(max_workers=2)

In [115]:
s = set()

In [9]:
def fn(i):
    return i

In [10]:
import math
fut1 = executor.submit(fn, 1)
fut1

<Future at 0x10c9f14d0 state=finished returned int>

In [118]:
x=fut1.result()

In [11]:
fut2 = executor.submit(fn, 2)
fut2

<Future at 0x10c9d2610 state=pending>

In [120]:
s.add(fut1)

In [121]:
s.add(fut2)

In [27]:
fut1.result()

1

In [12]:
s = {fut1, fut2}

In [15]:
done, pending_futures = wait(s, return_when='FIRST_COMPLETED')

In [16]:
done

{<Future at 0x10c9d2610 state=finished returned int>,
 <Future at 0x10c9f14d0 state=finished returned int>}

In [17]:
pending_futures

set()

In [122]:
from concurrent.futures import ThreadPoolExecutor, as_completed

completed_futures = as_completed(s)

In [123]:
s.add(executor.submit(fn, 3))

In [125]:
list(completed_futures)

[]

In [112]:
res = []

for fut in completed_futures:
    res.append(fut.result())

In [113]:
res

[2, 1]